In [ ]:
# !pip install ascon

In [13]:
import numpy as np
from tabulate import tabulate
import random 
import ascon

print("=======================================================================================")
print (" ")
print ("    < Example : 8-point DCT >    ")
print (" ")

def get_user_input():
    operation_numbers_input = input()
    operation_numbers = [int(size.strip()) for size in operation_numbers_input.split(', ') if size.strip().isdigit()]

    control_step_numbers_input = input()
    control_step_numbers = [int(size.strip()) for size in control_step_numbers_input.split(', ') if size.strip().isdigit()]

    return operation_numbers, control_step_numbers


def create_binary_table(operation_numbers, control_step_numbers):
    rows = max(control_step_numbers)
    cols = max(operation_numbers)

    # Initialize table (all zeros)
    table = np.zeros((rows, cols), dtype=int)

    # Fill table: mark 1 where operation belongs to control step
    for op, cs in zip(operation_numbers, control_step_numbers):
        if 1 <= cs <= rows and 1 <= op <= cols:
            table[cs - 1][op - 1] = 1

    # Prepare headers
    headers = [f"Opn{c+1}" for c in range(cols)]
    labeled_rows = [[f"CS{r+1}"] + list(table[r]) for r in range(rows)]

    # Store binary strings
    binary_values = ["".join(str(bit) for bit in row) for row in table]

    return table, binary_values, cols



if __name__ == "__main__":
    # Take runtime list input
    operation_numbers, control_step_numbers = get_user_input()
    # Create binary table
    table, binary_values, no_of_operations = create_binary_table(operation_numbers, control_step_numbers)

    # Display binary strings
    #print("\nAccessing Stored Binary Values Outside Function:=> ")
    print (" ")
    for i, val in enumerate(binary_values, start=1):
        print(f"CS{i}: {val}")


# Merge into a single string
merged_binary_values01 = "".join(binary_values)


# Binary ↔ Decimal conversion functions
def binary_to_decimal(binary_str):
    return int(binary_str, 2)

def decimal_to_binary(decimal_val, width):
    return format(decimal_val, f'0{width}b')  # zero-padded binary string


# XOR, Modulo, and Conversion (with per-row different keys)
def process_binary_values(binary_values, combined_key, highest_op):
    processed_binaries = []
    processed_decimals = []
    modulo_value = highest_op

    num_cs = len(binary_values)
    expected_key_length = num_cs * highest_op

    # Check combined key length
    if len(combined_key) != expected_key_length:
        raise ValueError(f"Total key length must be (No. of CS × No. of Ops) = {expected_key_length} bits")

    # Split combined key into per-CS slices
    key_slices = [combined_key[i * highest_op:(i + 1) * highest_op] for i in range(num_cs)]

    #print("\nKey slices used for each Control Step:")
    print (" ")
    for i, k in enumerate(key_slices, start=1):
        print(f"Key{i}: {k}")

    # Perform XOR + Modulo for each Control Step
    for bin_str, key_bin in zip(binary_values, key_slices):
        cs_list = [int(b) for b in bin_str]
        key_list = [int(k) for k in key_bin]
        # XOR
        xor_result = [cs ^ k for cs, k in zip(cs_list, key_list)]
        # Convert XOR to decimal
        xor_decimal = binary_to_decimal("".join(str(b) for b in xor_result))
        # Modulo (No. of Operations)
        mod_decimal = xor_decimal % modulo_value
        # Convert back to binary
        mod_binary = decimal_to_binary(mod_decimal, highest_op)
        processed_binaries.append(mod_binary)
        processed_decimals.append(mod_decimal)
    return processed_binaries, processed_decimals


print (" ")
total_key_length = len(binary_values) * no_of_operations
key_bin = input()

# key_bin = input(f"\nEnter total binary key K (length = {total_key_length})::=> ")
# key_bin = 110001010110100111101110110111010100101011010100010100101110011101110111100000100010011000011011100000101101001001101000 (as an Example)

# Or Generate random binary key of required length
# key_bin = ''.join(random.choice(['0', '1']) for _ in range(total_key_length))

if len(key_bin) != total_key_length:
    raise ValueError(f"Key length must be equal to total ({total_key_length}) bits "
                     f"(i.e., {len(binary_values)} control steps × {no_of_operations} operations)")

# Step 6: Process XOR, Modulo, and Binary conversion
final_binaries, final_decimals = process_binary_values(binary_values, key_bin, no_of_operations)

# If you want to Display Final processed binary values
# print("\nProcessed Binary Values After \"Decimal((CSX ⊕ KeyX) mod (HighestOp))\":=> ")
print (" ")

for i, val in enumerate(final_binaries, start=1):
    print(f"CS{i}: {val}")

# Merge into a single string
merged_binary_values02 = "".join(final_binaries)

# Concatenation of two Strings
concatenated_bitstream_01 = merged_binary_values01+merged_binary_values02

# print("\nConcatenated Bitstream 01:=> ")
# print(concatenated_bitstream_01)

concatenated_bitstream_02 = merged_binary_values02+merged_binary_values01

# print("\nConcatenated Bitstream 02:=> ")
# print(concatenated_bitstream_02)


# Binary → Hexadecimal conversion function
def binary_to_hexadecimal(concatenated_bitstream):
    # Convert binary string to integer, then format as uppercase hexadecimal
    return format(int(concatenated_bitstream, 2), 'X')

# print("\nOriginal Data/Plain Text for ASCON-128 Block:=> ")
original_data01 = binary_to_hexadecimal(concatenated_bitstream_01)
# print(original_data01)

# print("\nAssociated Data for ASCON-128 Block:=> ")
associated_data01 = binary_to_hexadecimal(concatenated_bitstream_02)
# print(associated_data01)


if len(original_data01) % 2 != 0:
    original_data01 = '0' + original_data01

if len(associated_data01) % 2 != 0:
    associated_data01 = '0' + associated_data01
    
# print("\nOriginal Data/Plain Text for ASCON-128 Block:=> ", original_data01)
# print("Length => ", len(original_data01))

# print("\nAssociated Data for ASCON-128 Block:=> ", associated_data01)
# print("Length => ", len(associated_data01))

# print("=======================================================================================")

def ascon128_encrypt(key: bytes, nonce: bytes, associated_data: bytes, plaintext: bytes) -> bytes:
    """
    ASCON-128 AEAD encryption.    Returns combined ciphertext+tag as bytes.
    """
    ciphertext = ascon.encrypt(key, nonce, associated_data, plaintext, variant="Ascon-128")
    return ciphertext

def ascon128_decrypt(key: bytes, nonce: bytes, associated_data: bytes, ciphertext: bytes) -> bytes:
    """
    ASCON-128 AEAD decryption.
    Accepts combined ciphertext+tag.
    """
    plaintext = ascon.decrypt(key, nonce, associated_data, ciphertext, variant="Ascon-128")
    return plaintext

print(" ")
# Example usage
hex_key = "3a16cbb97bd83b466f4ad3e30eeff611"
key = bytes.fromhex(hex_key)
print("Key : ", hex_key)

hex_nonce = "ae9ac672b8e6035c22f7bf26fea77c3f"
nonce = bytes.fromhex(hex_nonce)
print("Nonce : ", hex_nonce)

# Hex string
hex_pt = original_data01
pt = bytes.fromhex(hex_pt)      # Convert to bytes


# Hex string    
hex_ad = associated_data01
ad = bytes.fromhex(hex_ad)      # Convert to bytes


ct = ascon128_encrypt(key, nonce, ad, pt)
pt2 = ascon128_decrypt(key, nonce, ad, ct)

# print(" ")
# print("Original Data or Plain Text:=> ", pt.hex())
# print("Associated Data/Text:=> ", ad.hex())
# print(" ")

ciphertext_only = ct[:-16]
tag = ct[-16:]

ciphertext_only_in_hex = ciphertext_only.hex()
tag_only_in_hex = tag.hex()

print("\nCiphertext:=> ", ciphertext_only_in_hex)

print("\nTag:=>", tag_only_in_hex)

# print(" ")
ct_in_hex = ct.hex()
print("\nCiphertext||Tag:=> ", ct_in_hex)

# print("\nCiphertext||Tag Length (in Hex.):=>  ",len(ct_in_hex))

# print("=======================================================================================")

# Hexadecimal → Binary conversion function
def hex_to_binary(hex_str):
    # Each hex digit represents exactly 4 bits
    total_bits = len(hex_str) * 4
    # Convert hex → int → binary and pad with zeros to preserve MSB
    return format(int(hex_str, 16), f'0{total_bits}b')


generated_watermark_strength_in_hex = ct_in_hex

# print("\nGenerated watermark strength in hex:=> ", generated_watermark_strength_in_hex)

generated_watermark_strength_in_binary =  hex_to_binary(generated_watermark_strength_in_hex)

print("\nGenerated watermark strength in binary:=> ", generated_watermark_strength_in_binary)

ciphertext_only_in_binary =  hex_to_binary(ciphertext_only_in_hex)
tag_only_in_binary =  hex_to_binary(tag_only_in_hex)

ciphertext_only_in_binary =  hex_to_binary(ciphertext_only_in_hex)
tag_only_in_binary =  hex_to_binary(tag_only_in_hex)

print("\n Length of Final Watermark Strength:=>  ",len(generated_watermark_strength_in_binary))

print (" ")
print("=======================================================================================")


 
    < Example : 8-point DCT >    
 
1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15
1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 4, 5, 6, 7, 8
 
CS1: 111100000000000
CS2: 000011111000000
CS3: 000000000100000
CS4: 000000000010000
CS5: 000000000001000
CS6: 000000000000100
CS7: 000000000000010
CS8: 000000000000001
 
110001010110100111101110110111010100101011010100010100101110011101110111100000100010011000011011100000101101001001101000
 
Key1: 110001010110100
Key2: 111101110110111
Key3: 010100101011010
Key4: 100010100101110
Key5: 011101110111100
Key6: 000100010011000
Key7: 011011100000101
Key8: 101001001101000
 
CS1: 000000000001011
CS2: 000000000000011
CS3: 000000000001101
CS4: 000000000001011
CS5: 000000000001110
CS6: 000000000001110
CS7: 000000000000010
CS8: 000000000000111
 
Key :  3a16cbb97bd83b466f4ad3e30eeff611
Nonce :  ae9ac672b8e6035c22f7bf26fea77c3f

Ciphertext:=>  90ff7d6a6f8c1ae1db8e1f3ff587f11ee0acb7f496cd14ffc1e11816968d

Tag:=> ec43c6d8d92aa65ab2aa23e4308bd4d7

Ciphertext||Tag:=>  90ff